In [2]:
import os
import re
import pickle
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding,
    Bidirectional,   # wraps an LSTM layer to run it in both directions
    LSTM,
    Dense,
    Dropout,
)
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping

In [5]:
texts = [
# ------------------ Explain ------------------
"explain this", "can you explain this", "what does this mean",
"explain the concept", "help me understand this", "break this down",
"explain step by step", "clarify this topic", "what is this about",
"explain clearly", "explain in simple terms", "explain like im 5",
"give explanation", "detailed explanation please", "explain the idea",
"explain this topic", "i need an explanation", "explain properly",
"walk me through this", "explain it to me",

# ------------------ Summary ------------------
"summarize this", "give me a summary", "short summary please",
"brief this", "summarize the text", "give key points",
"tl dr", "main idea please", "summary in short",
"compress this text", "quick summary", "summarize briefly",
"what are the key points", "highlight main points",
"give overview", "summarize in one paragraph",
"make it short", "shorten this text", "condense this",
"give summary only",

# ------------------ Quiz ------------------
"give me a quiz", "create a quiz", "test me",
"ask me questions", "quiz me", "make a test",
"give practice questions", "check my knowledge",
"prepare a quiz", "generate questions", "exam questions please",
"practice quiz", "ask some questions", "test my understanding",
"give mcqs", "multiple choice questions", "quiz questions",
"create test questions", "give me questions",
"challenge me with questions",
]


labels = (
    [0]*20 +   # Explain
    [1]*20 +   # Summary
    [2]*20    # Quiz
)

In [6]:
def clean_text(t):
    t = t.lower()
    t = re.sub(r"[^a-zA-Z\s]", "", t)
    return t

texts = [clean_text(t) for t in texts]

In [7]:
tokenizer = Tokenizer(num_words=2000, oov_token="<OOV>")
tokenizer.fit_on_texts(texts)

sequences = tokenizer.texts_to_sequences(texts)

In [8]:
max_len = max(len(seq) for seq in sequences)
print(f" Vocabulary size : {len(tokenizer.word_index)}")
print(f" Max sequence len: {max_len}")

 Vocabulary size : 82
 Max sequence len: 5


In [9]:
X = pad_sequences(sequences, maxlen=max_len, padding="post")
y = np.array(labels)

In [10]:
os.makedirs("models", exist_ok=True)
with open("models/tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)
print("[INFO] Tokenizer saved → models/tokenizer.pkl")

[INFO] Tokenizer saved → models/tokenizer.pkl


In [11]:
with open("models/max_len.pkl", "wb") as f:
    pickle.dump(max_len, f)
print("[INFO] max_len saved → models/max_len.pkl")

[INFO] max_len saved → models/max_len.pkl


In [12]:
from tensorflow.keras import Input

In [13]:
model = Sequential([
    Input(shape=(max_len,)), 
    Embedding(input_dim=2000, output_dim=64, input_length=max_len),
    # Bidirectional(LSTM(64, return_sequences=False)),
    Bidirectional(
    LSTM(
        64,
        activation="tanh",
        recurrent_activation="sigmoid",
        use_bias=True,
        recurrent_dropout=0.2,   # 🔥 KEY LINE (forces non-CuDNN)
        unroll=False
    )
),
    Dropout(0.5),
    
    Dense(32, activation="relu"),
    Dropout(0.3),

    Dense(3, activation="softmax"),
])
model.summary()

c:\Users\anura\Downloads\AI_LEARN\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 5, 64)          │       128,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 128)            │        66,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         4,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 198,275 (774.51 KB)

 Trainable params: 198,275 (774.51 KB)

 Non-trainable params: 0 (0.00 B)

In [14]:
# model = Sequential([
#     Input(shape=(max_len,)),

#     Embedding(2000, 32),   # smaller

#     LSTM(32),              # no bidirectional

#     Dense(16, activation="relu"),

#     Dense(len(set(y)), activation="softmax")
# ])

In [15]:
model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"],
)


In [16]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True,
    verbose=1,
)

In [17]:
history = model.fit(
    X, y,
    epochs=100,
    batch_size=4,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1,
)

Epoch 1/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 5s 60ms/step - accuracy: 0.3750 - loss: 1.0920 - val_accuracy: 0.0000e+00 - val_loss: 1.1477
Epoch 2/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.5625 - loss: 1.0697 - val_accuracy: 0.0000e+00 - val_loss: 1.2333
Epoch 3/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.4375 - loss: 1.0399 - val_accuracy: 0.0000e+00 - val_loss: 1.3421
Epoch 4/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.4583 - loss: 1.0253 - val_accuracy: 0.0000e+00 - val_loss: 1.4949
Epoch 5/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.6042 - loss: 0.9570 - val_accuracy: 0.0000e+00 - val_loss: 1.6077
Epoch 6/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.5625 - loss: 0.8957 - val_accuracy: 0.0000e+00 - val_loss: 1.4669
Epoch 7/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.6875 - loss: 0.7403 - val_accuracy: 0.0000e+00 - val_loss: 1.4096
Epoch 8/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.8333 - los

In [18]:
model.save("models/advanced_intent_lstm.h5")
print("\n[DONE] Advanced Bidirectional LSTM saved → models/advanced_intent_lstm.h5")
print(f"       Final train accuracy : {history.history['accuracy'][-1]:.4f}")



[DONE] Advanced Bidirectional LSTM saved → models/advanced_intent_lstm.h5
       Final train accuracy : 1.0000
